# Project_Paru_CXR - Progress 4: Model Utama Swin-T (Google Colab)

Melatih model utama **Swin-T** (3 seed) dengan pembagian data yang sama dengan baseline, lalu membuat **laporan Progress 4** otomatis (perbandingan Swin-T vs EfficientNet-B0).
Hasil baseline dari Progress 3 di Google Drive dipakai ulang (tidak dilatih ulang).

**Langkah sebelum menjalankan:**
1. Menu **Runtime -> Change runtime type -> T4 GPU -> Save**.
2. Siapkan **token API Kaggle**: kaggle.com -> Settings -> **API Tokens** -> **Generate New Token**, lalu salin teks token yang muncul.
   - Simpan di **Colab Secrets**: klik ikon kunci di panel kiri Colab -> **Add new secret** -> Name `KAGGLE_API_TOKEN`, Value = token -> aktifkan *Notebook access*.
   - Alternatif: file `kaggle.json` (Legacy API Key) diunggah ke *My Drive*.
3. Menu **Runtime -> Run all**. Saat diminta, izinkan akses Google Drive.

Semua hasil (model, metrik, kurva, **laporan Progress 4 .docx**) disimpan di Google Drive: `My Drive/Project_Paru_CXR_hasil/`.
Jika Colab terputus, cukup **Run all** lagi: data PNG dipulihkan dari cache Drive (tanpa konversi ulang) dan run yang sudah selesai dilewati otomatis. Agar tidak terputus: biarkan tab tetap terbuka dan laptop tidak sleep.
Perkiraan waktu: pemulihan data dari cache Drive ±5 menit, pelatihan Swin-T 3 seed ±2-3 jam. Biarkan tab browser tetap terbuka.

In [ ]:
# 1. Cek GPU
import os, glob, shutil, subprocess, time, json
import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU belum aktif: Runtime -> Change runtime type -> T4 GPU -> Save, lalu Run all lagi.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__)

def run(cmd, check=True):
    """Jalankan perintah shell dan tampilkan output langsung."""
    print("\n$ " + cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Perintah gagal (kode {p.returncode}): {cmd}")
    return p.returncode

In [ ]:
# 2. Hubungkan Google Drive (hasil disimpan permanen di sini)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
OUT = f"{DRIVE}/Project_Paru_CXR_hasil"
for sub in ["runs", "results", "laporan"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
RSNA = "/content/rsna"
DATA_CACHE = f"{OUT}/data_png.tar"   # salinan PNG hasil konversi di Drive
print("Folder hasil:", OUT)
print("Cache data PNG di Drive:", "ADA" if os.path.exists(DATA_CACHE) else "belum ada (akan dibuat setelah konversi)")

In [ ]:
# 3. Token API Kaggle -> unduh data RSNA (±3,7 GB, beberapa menit)
# Opsi token baru (bila Kaggle memberi token teks, bukan file): tempel di antara tanda kutip.
KAGGLE_API_TOKEN = ""

# Cara paling aman: simpan token di Colab Secrets (ikon kunci di panel kiri) dengan nama KAGGLE_API_TOKEN
if not KAGGLE_API_TOKEN:
    try:
        from google.colab import userdata
        KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN") or ""
    except Exception:
        pass

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kj = os.path.expanduser("~/.kaggle/kaggle.json")
if KAGGLE_API_TOKEN:
    print("Memakai token API Kaggle (format baru)")
    os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
elif os.path.exists(f"{DRIVE}/kaggle.json"):
    shutil.copy(f"{DRIVE}/kaggle.json", kj)
elif not os.path.exists(kj):
    from google.colab import files
    print("kaggle.json tidak ditemukan di Drive. Pilih file kaggle.json dari komputer Anda:")
    up = files.upload()
    with open(kj, "wb") as f:
        f.write(up[next(iter(up))])
if os.path.exists(kj):
    os.chmod(kj, 0o600)
run("pip install -q -U kaggle")

if os.path.exists(DATA_CACHE):
    print("Cache PNG ditemukan di Drive -> unduh RSNA dilewati")
elif not os.path.exists(f"{RSNA}/stage_2_train_labels.csv"):
    os.makedirs(RSNA, exist_ok=True)
    run(f"kaggle competitions download -c rsna-pneumonia-detection-challenge -p {RSNA}")
    z = glob.glob(f"{RSNA}/*.zip")[0]
    run(f"unzip -q -o '{z}' -x 'stage_2_test_images/*' -d {RSNA}")
    os.remove(z)
if not os.path.exists(DATA_CACHE):
    print("RSNA:", RSNA, "| jumlah DICOM:", len(glob.glob(f"{RSNA}/stage_2_train_images/*.dcm")))

In [ ]:
# 4. Ambil kode dari GitHub + pustaka; folder hasil diarahkan ke Google Drive
REPO = "/content/Project_Paru_CXR"
if os.path.isdir(f"{REPO}/.git"):
    run(f"git -C {REPO} pull --ff-only")
else:
    run(f"git clone --depth 1 https://github.com/monicabortang/Project_Paru_CXR.git {REPO}")
os.chdir(REPO)
run("pip install -q timm pydicom python-docx")
for sub in ["runs", "results", "laporan"]:
    p = f"{REPO}/{sub}"
    if os.path.islink(p):
        continue
    if os.path.isdir(p):
        shutil.copytree(p, f"{OUT}/{sub}", dirs_exist_ok=True)
        shutil.rmtree(p)
    os.symlink(f"{OUT}/{sub}", p)
print("Folder kerja:", os.getcwd(), "| runs/results/laporan -> Google Drive")

In [ ]:
# 5. Konversi DICOM -> PNG 512 px + metadata (±25-40 menit di Colab)
DATA = "/content/data"
if not os.path.exists(f"{DATA}/metadata.csv"):
    if os.path.exists(DATA_CACHE):
        print("Memulihkan PNG dari cache Google Drive (±3-5 menit) ...")
        run(f"tar -xf '{DATA_CACHE}' -C /content")
    else:
        if not os.path.exists(f"{RSNA}/stage_2_train_labels.csv"):
            raise SystemExit("Data RSNA belum terunduh: sel 3 gagal. Periksa pesan error di sel 3 "
                             "(biasanya token KAGGLE_API_TOKEN), perbaiki, lalu jalankan dari sel 3.")
        run(f"python -m src.prepare_data --rsna_dir {RSNA} --out_dir {DATA} --size 512 --workers {os.cpu_count()}")
        print("Menyimpan cache PNG ke Google Drive agar tidak perlu konversi ulang ...")
        run(f"tar -cf '{DATA_CACHE}.tmp' -C /content data && mv '{DATA_CACHE}.tmp' '{DATA_CACHE}'")
n_png = len(glob.glob(f"{DATA}/images/*.png"))
print("Jumlah PNG:", n_png)
assert n_png >= 26000, "Konversi belum lengkap"

In [ ]:
# 6. Pakai split dari repositori (sama dengan laporan Progress 2) dan cek kecocokannya
import pandas as pd
SPLITS = f"{REPO}/splits"
meta_ids = set(pd.read_csv(f"{DATA}/metadata.csv").patientId)
for f in ["S1_all/train.csv", "S1_all/val.csv", "test/all.csv"]:
    ids = set(pd.read_csv(f"{SPLITS}/{f}").patientId)
    miss = len(ids - meta_ids)
    print(f"{f:20s} n={len(ids):6d}  tidak ditemukan={miss}")
    assert miss == 0

In [ ]:
# 7. (Opsional) Uji eksternal Kermany - dilewati otomatis bila gagal
USE_KERMANY = True
EXTARG = ""
EXT = "/content/external/kermany"
if USE_KERMANY:
    try:
        if not os.path.exists(f"{EXT}/kermany.csv"):
            run("kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/kermany_zip")
            run("unzip -q -o /content/kermany_zip/*.zip -d /content/kermany")
            src = [c for c in glob.glob("/content/kermany/**/chest_xray", recursive=True)
                   if os.path.isdir(f"{c}/test/NORMAL")][0]
            run(f"python -m src.external kermany --src '{src}' --out {EXT} --subsets test")
        EXTARG = f"--external kermany={EXT}/kermany.csv"
    except Exception as e:
        print("Kermany dilewati:", e)
print("Argumen eksternal:", EXTARG or "(tidak ada)")

In [ ]:
# 8. Smoke test Swin-T di GPU (±3 menit)
for v in ("run", "REPO", "DATA", "SPLITS", "EXTARG"):
    if v not in globals():
        raise SystemExit("Sesi Colab terputus/sel sebelumnya belum jalan. Pilih Runtime -> Run all.")
os.chdir(REPO)
W = os.cpu_count()
run(f"python scripts/run_experiments.py --group smoke --models swin_tiny_patch4_window7_224 --splits {SPLITS} --img_root {DATA} --workers {W}")

In [ ]:
# 9. PELATIHAN MODEL UTAMA: Swin-T, S1, seed 0/1/2 (±2-3 jam)
#    Grup 'baseline' dijalankan dulu: bila hasil Progress 3 sudah ada di Drive, otomatis dilewati.
t0 = time.time()
run(f"python scripts/run_experiments.py --group baseline --splits {SPLITS} --img_root {DATA} --workers {W} {EXTARG}")
run(f"python scripts/run_experiments.py --group main --splits {SPLITS} --img_root {DATA} --workers {W} {EXTARG}")
print(f"Pelatihan selesai dalam {(time.time() - t0) / 60:.1f} menit")

In [ ]:
# 10. Rekap + LAPORAN PROGRESS 4 otomatis (dan Progress 3 bila belum ada)
run("python -m src.summarize --results results/all_results.csv --runs runs --out results/tables")
if not os.path.exists("laporan/08_Progress 3.docx"):
    run('python scripts/make_progress3_report.py --platform "Google Colab (GPU NVIDIA T4)" --out "laporan/08_Progress 3.docx"')
run('python scripts/make_progress4_report.py --platform "Google Colab (GPU NVIDIA T4)" --out "laporan/08_Progress 4.docx"')
from IPython.display import Image, display
r = pd.read_csv("results/all_results.csv")
t = r[r.run_tag == "S1_all"].groupby(["model", "test"])[["auc", "macro_f1", "sensitivity", "specificity"]].mean().round(4)
display(t)
display(Image("laporan/gambar_progress4/auc_banding.png"))
print("\nSELESAI. Buka Google Drive -> My Drive/Project_Paru_CXR_hasil/laporan/08_Progress 4.docx")